In [2]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='Parrots don’t “talk” in the same way humans do—they’re superb **imitators**. Their ability to copy sounds, including human speech, comes from a combination of anatomy, brain wiring, and social instincts:\n\n| Factor | What it means for “talking” |\n|--------|-----------------------------|\n| **Vocal anatomy** | Parrots have a highly flexible syrinx (the bird equivalent of a voice box) and a muscular tongue that can shape a wide range of sounds, from whistles to human‑like syllables. |\n| **Brain structure** | The *nidopallium* (especially the “song system” within it) is analogous to the human speech‑processing areas. It lets them learn, store, and reproduce complex acoustic patterns. |\n| **Social nature** | In the wild, parrots live in tight‑knit flocks where constant vocal exchange maintains group cohesion. Mimicking sounds is a way to “fit in” and bond with flock mates—so a pet parrot treats you as part of its flock. |\n| **Reinforcement** | They are excellent at 

In [3]:

from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [4]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'User asks for weather in Boston. Use function get_weather.', 'tool_calls': [{'id': 'fc_ef36a755-d8f3-46f6-a5d9-15655da1177d', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 127, 'total_tokens': 167, 'completion_time': 0.084194043, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.005809425, 'prompt_tokens_details': None, 'queue_time': 0.28115666, 'total_time': 0.090003468}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_017482bd7f', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f9d5e-da58-7240-ba28-34ea92205f80-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_ef36a755-d8f3-46f6-a5d9-15655da1177d', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens

## Tool Execution

In [5]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

It's sunny in Boston.


In [6]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks: "What\'s the weather in Boston?" Need to call get_weather function with location "Boston".', 'tool_calls': [{'id': 'fc_c95d8831-f694-48c2-b113-5439c9ba4c01', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 126, 'total_tokens': 175, 'completion_time': 0.103188263, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.028138509, 'prompt_tokens_details': None, 'queue_time': 0.294365727, 'total_time': 0.131326772}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_87b3c396db', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f9d5f-4bd7-7d60-8db6-63b5850d18e3-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 